# Rods

---

Not all particles are spheres. If we want to simulate these particles we have two options:
- we can change the interaction potential to be non-spherical 
- we can chain particles rigidly so the combined interaction potentials are non-spherical

We will start with the latter

In [1]:
import hoomd # simulation engine
import numpy as np # arrays and random numbers
import gsd.hoomd # reading and writing HOOMD trajectory files
import os # deleting old files

### Rod shape and density
- Each rod has length $(n-1)b + \sigma$ and width $\sigma$; its aspect ratio (length / width) is one of the main things that controls how rods behave
- We set the density with the area fraction $\phi$ (fraction of the box covered by rods) and work out the box size from it
- Keep the box many rod lengths across, otherwise rods feel their own periodic images (finite-size effects)

In [16]:
N = 100 # number of rods
n = 5 # number of beads in each rod
b = 0.5 # spacing between beads in a rod
phi = 0.2 # area fraction: fraction of the box covered by rods
kT = 1.0 # temperature (thermal energy)

In [17]:
# shape of one rod in its own frame: n beads along the x axis, centred on the origin
bead_x = (np.arange(n) - (n - 1) / 2) * b # x position of each bead
bead_positions = [(x, 0.0, 0.0) for x in bead_x]
bead_types = ['A'] * (n - 1) + ['H'] # the last bead (at the +x end) is the head H, so the rod points towards its head
rod_length = (n - 1) * b # distance from the first bead centre to the last
I_z = np.sum(bead_x**2) # moment of inertia about z (each bead has mass 1)

a_rod = rod_length * 1 + np.pi / 4 # area of one rod: a strip 1 wide plus two half-disc end caps
L = np.sqrt(N * a_rod / phi) # size of our 2D box, chosen to give area fraction phi
A = L**2 # Area of our 2D box

### Starting configuration
- Starting on a lattice avoids overlaps even when dense, but all the rods start lined up
- Run for a while and check the rods have forgotten the lattice (equilibration) before measuring anything

In [18]:
# place N rods on a lattice: rows of rods lying along the x axis
nx = max(1, int(round(np.sqrt(N / (rod_length + 1))))) # rods per row, so the gaps along and across the rods scale with the rod shape
ny = int(np.ceil(N / nx)) # number of rows
dx = L / nx # spacing between rod centres along a row
dy = L / ny # spacing between rows
if dx < rod_length + 1 or dy < 1:
    raise ValueError("phi is too high for this lattice, the rods would overlap")

i, j = np.meshgrid(np.arange(nx), np.arange(ny)) # lattice site indices
positions = np.zeros((N, 3)) # x, y, z of each rod centre (z stays 0 in 2D)
positions[:, 0] = (i.ravel()[:N] + 0.5) * dx - L / 2 # x in [-L/2, L/2)
positions[:, 1] = (j.ravel()[:N] + 0.5) * dy - L / 2 # y in [-L/2, L/2)

# point each rod along +x or -x at random, so they don't all face the same way
rng = np.random.default_rng() # random number generator
theta = rng.choice([0, np.pi], size=N) # angle of each rod
orientations = np.zeros((N, 4)) # quaternion for a rotation by theta about z
orientations[:, 0] = np.cos(theta / 2)
orientations[:, 3] = np.sin(theta / 2)

In [19]:
# build the initial frame: only the rod centres, HOOMD adds the beads later
frame = gsd.hoomd.Frame()
frame.particles.N = N
frame.particles.types = ['R', 'A', 'H'] # R = rod centre, A = body bead, H = head bead (A and H must be listed even though there are none yet)
frame.particles.typeid = np.zeros(N, dtype=int) # every particle is a rod centre (type R)
frame.particles.position = positions
frame.particles.orientation = orientations
frame.particles.mass = n * np.ones(N) # mass of the whole rod
frame.particles.moment_inertia = np.tile([0, 0, I_z], (N, 1)) # only rotation about z in 2D
frame.configuration.box = [L, L, 0, 0, 0, 0] # Lx, Ly, Lz, xy, xz, yz (Lz = 0 makes it 2D)

In [20]:
# delete any old initial config file
init_filename = "rods_init.gsd"
if os.path.exists(init_filename):
    os.remove(init_filename)

# save the initial config ("x" creates a new file)
with gsd.hoomd.open(name=init_filename, mode="x") as f:
    f.append(frame)

In [21]:
GPU = hoomd.device.GPU() # use CPU() instead if you don't have a GPU
simulation = hoomd.Simulation(device=GPU, seed=1) # seed sets the random numbers
simulation.create_state_from_gsd(filename=init_filename) # load the rod centres

### Rigid bodies
- Each rod is a central particle `R` plus `n` beads; HOOMD only moves `R`, and the beads are carried along with it
- Forces on the beads are added up into a force and a torque on `R`, which is what moves and turns the rod

In [22]:
# define the rod shape and add the beads to every rod
rigid = hoomd.md.constrain.Rigid()
rigid.body['R'] = {
    "constituent_types": bead_types, # body beads are type A, the head bead is type H
    "positions": bead_positions, # bead positions relative to the rod centre
    "orientations": [(1.0, 0.0, 0.0, 0.0)] * n, # beads are not rotated relative to the rod
}
rigid.create_bodies(simulation.state) # adds n beads to every rod, only call this once

In [23]:
integrator = hoomd.md.Integrator(dt=1e-4, integrate_rotational_dof=True) # rotational_dof lets the rods rotate
integrator.rigid = rigid # keep the beads fixed in each rod

### Rotational diffusion
- Rods also get random rotational kicks, so their direction diffuses with $D_r = k_BT/\gamma_r$
- Simplification: HOOMD uses the same drag in every direction, while real rods slide about twice as easily along their length as sideways

In [24]:
# Brownian dynamics for the rod centres only, the beads just follow their rod
Brownian = hoomd.md.methods.Brownian(filter=hoomd.filter.Rigid(("center", "free")), kT=kT)
Brownian.gamma['R'] = n # translational drag: each bead adds drag 1
Brownian.gamma_r['R'] = (1.0, 1.0, I_z) # rotational drag: each bead adds x_i^2 (x and y must not be 0, but are unused in 2D)
integrator.methods.append(Brownian)

In [25]:
cell = hoomd.md.nlist.Cell(buffer=0.4, exclusions=('bond', 'body')) # 'body': beads in the same rod don't interact

### WCA interaction
- WCA is the LJ potential cut off at its minimum ($r = 2^{1/6}\sigma$) and shifted up to zero, so it is purely repulsive
- It just stops rods overlapping (excluded volume), with no attraction

In [26]:
# WCA (purely repulsive LJ) between beads, set r_cut = 2.5 instead for attractive LJ
# the head H interacts the same as the body beads A, change the H pairs to give the head different interactions
lj = hoomd.md.pair.LJ(nlist=cell, mode='shift')
lj.params[(['A', 'H'], ['A', 'H'])] = dict(epsilon=1.0, sigma=1.0) # A-A, A-H and H-H
lj.r_cut[(['A', 'H'], ['A', 'H'])] = 2**(1/6)
lj.params[('R', ['R', 'A', 'H'])] = dict(epsilon=0.0, sigma=1.0) # rod centres don't interact
lj.r_cut[('R', ['R', 'A', 'H'])] = 0.0
integrator.forces.append(lj)

In [27]:
simulation.operations.integrator = integrator # attach the integrator to the simulation

In [28]:
# delete any old trajectory file
trajectory_filename = "rods_trajectory.gsd"
if os.path.exists(trajectory_filename):
    os.remove(trajectory_filename)

# save the trajectory to file
gsd_writer = hoomd.write.GSD(filename=trajectory_filename,
                            trigger=hoomd.trigger.Periodic(period=1000), # save a frame every 1000 steps
                            mode="wb", # overwrite any existing file
                            filter=hoomd.filter.All(), # save rod centres and beads
                            dynamic=['property', 'particles/image']) # save positions, orientations and box crossings every frame
simulation.operations.writers.append(gsd_writer) # attach the writer to the simulation

In [29]:
simulation.run(100000) # run for 100,000 steps
gsd_writer.flush() # write any frames still held in memory to the file